In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# MemoryBandit MVGC (RPE Categories)\n
\n
Replicates the hit/miss MVGC pipeline but splits by `rpe_cat` (`neg`, `neutral`, `pos`).


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
import seaborn as sns

from membandit.plotting_style import set_plot_style
from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit.io import load_subject_epochs
from membandit.granger import run_or_load_subject_gc_rpe_df, load_gc_rpe_df_cache, merge_with_behavior_rpe, validate_smoke_outputs_rpe, generate_subject_plot_rpe, plot_smoke_rpe, plot_aggregate_rpe, fit_mixed_effects_rpe, plot_mixed_effects_rpe

warnings.filterwarnings('ignore')
set_plot_style()



In [ ]:

# Paths
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvgc_rpe'
plot_savepath = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-MultivariateGrangerCausality_RPE'
os.makedirs(savepath, exist_ok=True)
os.makedirs(plot_savepath, exist_ok=True)

# Subjects from demographics (same pattern as original notebook)
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().unique().tolist())

# Load + preprocess behavior
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print('Subjects:', len(subjects))
print('learn_df:', learn_df.shape)
print('combined_df:', combined_df.shape)
print('learn_df columns check:', {'participant','trials','rpe'}.issubset(learn_df.columns))



In [ ]:
combined_df.loc[(combined_df.condition == 'Day1') & (combined_df.recog_rt>1)]

In [ ]:
# Analysis configuration
ANALYSIS = {
    'event': 'cue_on',
    'day': 'Day1',
    'tmin': 0.0,
    'tmax': 1.0,
    'target_sfreq': 50,
    'fmin': 4,
    'fmax': 8,
    'gc_n_lags': 10,
    'n_freqs': None,
    'min_trials': 5,
}

# Surrogate configuration (source-shuffle null)
USE_SURROGATES = True
N_SURROGATES_SMOKE = 100
N_SURROGATES_BATCH = 100
SURROGATE_TAIL = 'greater'   # one-sided
SURROGATE_RANDOM_STATE = 42
N_JOBS_SURROGATES_SMOKE = 12
N_JOBS_SURROGATES_BATCH = 12
SURROGATE_PARALLEL_BACKEND = 'loky'

# DataFrame cache configuration
GC_RPE_DF_CACHE_DIR = os.path.join(savepath, 'gc_df_cache_rpe')
OVERWRITE_GC_RPE_DF_CACHE = True

ROI_PAIRS = [
    ('HPC', 'ACC'),
    ('HPC', 'AMY'),
    ('HPC', 'dlPFC'),
    ('HPC', 'dmPFC'),
    ('HPC', 'OFC'),
    ('ACC', 'dmPFC'),
    ('ACC', 'dlPFC'),
    ('ACC', 'OFC'),
    ('ACC', 'AMY'),

]

RUN_SINGLE_SUBJECT_SMOKE = True
SMOKE_SUBJECT = 'UI002' if 'UI002' in subjects else subjects[0]
SMOKE_ROI_SOURCE, SMOKE_ROI_TARGET = ROI_PAIRS[0]

print('Smoke subject:', SMOKE_SUBJECT)
print('Smoke pair:', SMOKE_ROI_SOURCE, '->', SMOKE_ROI_TARGET)
print('Surrogates enabled:', USE_SURROGATES)
print('Smoke surrogates:', N_SURROGATES_SMOKE, '| Batch surrogates:', N_SURROGATES_BATCH)
print('GC RPE DF cache dir:', GC_RPE_DF_CACHE_DIR)
print('Overwrite cache:', OVERWRITE_GC_RPE_DF_CACHE)
rpe_behavior_df = combined_df if ANALYSIS['event'] == 'cue_on' else learn_df
print('RPE behavior source:', 'combined_df' if ANALYSIS['event'] == 'cue_on' else 'learn_df')

ANALYSIS_RESULTS = globals().get('ANALYSIS_RESULTS', {})
SMOKE_RESULTS = globals().get('SMOKE_RESULTS', {})


def _analysis_value_token(value):
    return str(value).replace('.', 'p').replace('-', 'm')


def _normalize_roi_pairs(pairs):
    return sorted([list(pair) for pair in pairs])


def _build_analysis_tag(analysis, roi_pairs):
    roi_pairs_norm = _normalize_roi_pairs(roi_pairs)
    roi_hash = hashlib.sha1(json.dumps(roi_pairs_norm, sort_keys=True).encode('utf-8')).hexdigest()[:10]
    band_tag_local = f"{analysis['fmin']}-{analysis['fmax']}Hz"
    return (
        f"{analysis['event']}_{analysis['day']}"
        f"_t{_analysis_value_token(analysis['tmin'])}-{_analysis_value_token(analysis['tmax'])}"
        f"_sf{_analysis_value_token(analysis['target_sfreq'])}"
        f"_{band_tag_local}"
        f"_lag{_analysis_value_token(analysis['gc_n_lags'])}"
        f"_roi{roi_hash}"
    )


analysis_tag = _build_analysis_tag(ANALYSIS, ROI_PAIRS)
fmin_tag = f"{ANALYSIS['fmin']}Hz"
band_tag = f"{ANALYSIS['fmin']}-{ANALYSIS['fmax']}Hz"
ANALYSIS_DIR = os.path.join(savepath, 'notebook_runs', analysis_tag)
os.makedirs(ANALYSIS_DIR, exist_ok=True)


def get_analysis_paths(tag=None, source_roi=None):
    tag = analysis_tag if tag is None else tag
    base_dir = os.path.join(savepath, 'notebook_runs', tag)
    os.makedirs(base_dir, exist_ok=True)
    paths = {
        'base_dir': base_dir,
        'all_gc_csv': os.path.join(base_dir, 'all_pairs_mvgc_rpe.csv'),
        'cache_log_csv': os.path.join(base_dir, 'cache_log_rpe.csv'),
        'merged_gc_csv': os.path.join(base_dir, 'merged_gc_rpe.csv'),
        'smoke_df_csv': os.path.join(base_dir, 'smoke_gc_rpe.csv'),
        'smoke_meta_json': os.path.join(base_dir, 'smoke_gc_rpe_meta.json'),
    }
    if source_roi is not None:
        paths['stats_csv'] = os.path.join(base_dir, f'mixed_model_gc_rpe_stats_{source_roi}.csv')
    return paths


analysis_paths = get_analysis_paths()
print('Analysis tag:', analysis_tag)
print('Analysis dir:', ANALYSIS_DIR)
print('Filename band tag:', band_tag)


In [ ]:
# Trialwise theta example configuration
EXAMPLE_PAIR = ('HPC', 'ACC')
EXAMPLE_CONDITION = 'pos'
EXAMPLE_POWER_PERCENTILE = 50
EXAMPLE_MAX_LAG_MS = 150
EXAMPLE_TIME_WINDOW = (ANALYSIS['tmin'], ANALYSIS['tmax'])
EXAMPLE_REQUIRE_HPC_LEADS = True
EXAMPLE_TOP_K_CANDIDATES = 3

print('Example pair:', EXAMPLE_PAIR[0], '->', EXAMPLE_PAIR[1])
print('Example condition:', EXAMPLE_CONDITION if EXAMPLE_CONDITION is not None else 'any')
print('Example power percentile:', EXAMPLE_POWER_PERCENTILE)
print('Example max lag (ms):', EXAMPLE_MAX_LAG_MS)
print('Require HPC leading ACC:', EXAMPLE_REQUIRE_HPC_LEADS)
print('Top candidate plots:', EXAMPLE_TOP_K_CANDIDATES)




In [ ]:
# Smoke run + validation + smoke plot
smoke_pack = SMOKE_RESULTS.get(analysis_tag)
if smoke_pack is not None:
    print('Loaded smoke pack from in-memory registry for', analysis_tag)
elif RUN_SINGLE_SUBJECT_SMOKE:
    smoke_df, smoke_result, smoke_meta = run_or_load_subject_gc_rpe_df(
        subj_id=SMOKE_SUBJECT,
        roi_source=SMOKE_ROI_SOURCE,
        roi_target=SMOKE_ROI_TARGET,
        base_dir=base_dir,
        learn_df=rpe_behavior_df,
        cache_dir=GC_RPE_DF_CACHE_DIR,
        overwrite_cache=OVERWRITE_GC_RPE_DF_CACHE,
        return_result_dict=True,
        event=ANALYSIS['event'],
        day=ANALYSIS['day'],
        tmin=ANALYSIS['tmin'],
        tmax=ANALYSIS['tmax'],
        target_sfreq=ANALYSIS['target_sfreq'],
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
        gc_n_lags=ANALYSIS['gc_n_lags'],
        n_freqs=ANALYSIS['n_freqs'],
        min_trials=ANALYSIS['min_trials'],
        use_surrogates=USE_SURROGATES,
        n_surrogates=N_SURROGATES_SMOKE,
        surrogate_tail=SURROGATE_TAIL,
        random_state=SURROGATE_RANDOM_STATE,
        n_jobs_surrogates=N_JOBS_SURROGATES_SMOKE,
        surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
        store_surrogates=True,
        verbose=True,
    )
    if smoke_result is None:
        raise RuntimeError('Smoke compute did not return a result_dict')

    smoke_fig = generate_subject_plot_rpe(
        subj_id=SMOKE_SUBJECT,
        roi_source=SMOKE_ROI_SOURCE,
        roi_target=SMOKE_ROI_TARGET,
        result_dict=smoke_result,
        epoch_data=smoke_result['epoch_data'],
        learn_df=rpe_behavior_df,
        fmin=ANALYSIS['fmin'],
        fmax=ANALYSIS['fmax'],
        event=ANALYSIS['event'],
        savepath=analysis_paths['base_dir'],
    )

    smoke_pack = {
        'result': smoke_result,
        'smoke_df': smoke_df,
        'fig': smoke_fig,
        'cache_meta': smoke_meta,
    }
    SMOKE_RESULTS[analysis_tag] = smoke_pack
    ANALYSIS_RESULTS.setdefault(analysis_tag, {})['smoke_pack'] = smoke_pack
    smoke_df.to_csv(analysis_paths['smoke_df_csv'], index=False)
    with open(analysis_paths['smoke_meta_json'], 'w') as f:
        json.dump(smoke_meta, f, indent=2, default=str)

if RUN_SINGLE_SUBJECT_SMOKE and smoke_pack is not None:
    validate_smoke_outputs_rpe(smoke_pack, rpe_behavior_df, event=ANALYSIS['event'])
    print('Smoke cache hit:', smoke_pack['cache_meta'].get('cache_hit'), '| file:', smoke_pack['cache_meta'].get('cache_file'))
    print('Smoke rows:', len(smoke_pack['smoke_df']))
    if 'gc_net_z' in smoke_pack['smoke_df'].columns:
        print('Smoke gc_net_z range:',
              float(np.nanmin(smoke_pack['smoke_df']['gc_net_z'])),
              float(np.nanmax(smoke_pack['smoke_df']['gc_net_z'])))
    display(smoke_pack['smoke_df'].head())
    plot_smoke_rpe(smoke_pack)


In [ ]:
# Trialwise theta example helpers
def _safe_zscore(x):
    x = np.asarray(x, dtype=float)
    mean_x = np.nanmean(x)
    std_x = np.nanstd(x)
    if (not np.isfinite(std_x)) or std_x == 0:
        return np.zeros_like(x, dtype=float)
    return (x - mean_x) / std_x


def load_native_example_epoch_data(subj_id, roi_source, roi_target, event, day, tmin, tmax, fmin, fmax, use_top_electrode_only=False):
    epoch_data = load_subject_epochs(
        subj_id=subj_id,
        base_dir=base_dir,
        event=event,
        roi_source=roi_source,
        roi_target=roi_target,
        tmin=tmin,
        tmax=tmax,
        target_sfreq=1e9,
        day=day,
        fmin=fmin,
        fmax=fmax,
        use_top_electrode_only=use_top_electrode_only,
    )
    if epoch_data is None:
        raise RuntimeError(
            f'Unable to load native-rate epochs for {subj_id} {roi_source}->{roi_target} ({event}, {day}).'
        )
    if not np.isclose(float(epoch_data['sfreq']), float(epoch_data['original_sfreq'])):
        raise RuntimeError(
            'Native example epoch load unexpectedly returned resampled data: '
            f"sfreq={epoch_data['sfreq']} original_sfreq={epoch_data['original_sfreq']}"
        )
    return epoch_data


def _summarize_gc_example_context(result_dict, preferred_condition=None):
    epoch_data = result_dict['epoch_data']
    rows = []
    for hemi in ('l', 'r'):
        source_channels = list(epoch_data.get(f'source_channels_{hemi}', []))
        target_channels = list(epoch_data.get(f'target_channels_{hemi}', []))
        if len(source_channels) == 0 or len(target_channels) == 0:
            continue

        hemi_data = result_dict.get('gc_by_hemi', {}).get(hemi, {})
        for cond, cond_data in hemi_data.items():
            if preferred_condition is not None and cond != preferred_condition:
                continue
            if (not isinstance(cond_data, dict)) or (not cond_data.get('success', False)):
                continue

            metric_key = 'gc_net_z' if 'gc_net_z' in cond_data else 'gc_net'
            metric = np.asarray(cond_data.get(metric_key, []), dtype=float)
            if metric.size == 0 or np.all(np.isnan(metric)):
                continue

            trial_indices = [int(x) for x in cond_data.get('trial_indices', [])]
            trial_value_col = result_dict.get('gc_by_hemi', {}).get('trial_value_col', 'trials')
            trial_values = cond_data.get(trial_value_col, [])
            if len(trial_values) != len(trial_indices):
                trial_values = [idx + 1 for idx in trial_indices]

            rows.append({
                'hemisphere': hemi,
                'condition': cond,
                'metric_key': metric_key,
                'peak_gc': float(np.nanmax(metric)),
                'mean_gc': float(np.nanmean(metric)),
                'n_trials': int(cond_data.get('n_trials', len(trial_indices))),
                'trial_indices': trial_indices,
                'trial_values': [int(v) for v in trial_values],
                'trial_value_col': trial_value_col,
                'source_channels': source_channels,
                'target_channels': target_channels,
            })

    if len(rows) == 0:
        return pd.DataFrame()

    gc_context_df = pd.DataFrame(rows)
    gc_context_df = gc_context_df.sort_values(['peak_gc', 'mean_gc', 'n_trials'], ascending=[False, False, False]).reset_index(drop=True)
    return gc_context_df


def extract_theta_pair_traces(epoch_data, hemisphere, fmin, fmax, trial_indices=None, trial_values=None, time_window=None):
    source_channels = list(epoch_data.get(f'source_channels_{hemisphere}', []))
    target_channels = list(epoch_data.get(f'target_channels_{hemisphere}', []))
    if len(source_channels) == 0 or len(target_channels) == 0:
        raise ValueError(f'Missing source/target channels for hemisphere {hemisphere}')

    epochs = epoch_data['epochs']
    source_epochs = epochs.copy().pick(source_channels).filter(fmin, fmax, verbose=False)
    target_epochs = epochs.copy().pick(target_channels).filter(fmin, fmax, verbose=False)

    source_traces = source_epochs.get_data(copy=True).mean(axis=1)
    target_traces = target_epochs.get_data(copy=True).mean(axis=1)
    times = np.asarray(epoch_data['times'], dtype=float)

    if time_window is not None:
        time_mask = (times >= time_window[0]) & (times <= time_window[1])
        if not np.any(time_mask):
            raise ValueError('Requested EXAMPLE_TIME_WINDOW does not overlap epoch times')
        times = times[time_mask]
        source_traces = source_traces[:, time_mask]
        target_traces = target_traces[:, time_mask]

    if trial_indices is None:
        trial_indices = np.arange(source_traces.shape[0], dtype=int)
    else:
        trial_indices = np.asarray(trial_indices, dtype=int)
        valid_mask = (trial_indices >= 0) & (trial_indices < source_traces.shape[0])
        trial_indices = trial_indices[valid_mask]

    if trial_indices.size == 0:
        raise ValueError('No valid trial indices available for theta example extraction')

    source_traces = source_traces[trial_indices]
    target_traces = target_traces[trial_indices]

    if trial_values is None:
        trial_values = trial_indices + 1
    else:
        trial_values = np.asarray(trial_values, dtype=int)
        if trial_values.shape[0] != trial_indices.shape[0]:
            trial_values = trial_indices + 1

    return {
        'hemisphere': hemisphere,
        'source_channels': source_channels,
        'target_channels': target_channels,
        'source_traces': source_traces,
        'target_traces': target_traces,
        'times': times,
        'sfreq': float(epoch_data['sfreq']),
        'original_sfreq': float(epoch_data.get('original_sfreq', epoch_data['sfreq'])),
        'trial_indices': trial_indices,
        'trial_values': trial_values,
    }


def _peak_cross_correlation(source_trace, target_trace, max_lag_samples):
    source_z = _safe_zscore(source_trace)
    target_z = _safe_zscore(target_trace)
    full_corr = correlate(target_z, source_z, mode='full', method='auto')
    denom = np.sqrt(np.sum(source_z ** 2) * np.sum(target_z ** 2))
    if (not np.isfinite(denom)) or denom == 0:
        return np.nan, 0
    full_corr = full_corr / denom
    lags = correlation_lags(target_z.size, source_z.size, mode='full')
    lag_mask = np.abs(lags) <= max_lag_samples
    if not np.any(lag_mask):
        return np.nan, 0
    corr_window = full_corr[lag_mask]
    lag_window = lags[lag_mask]
    if np.all(np.isnan(corr_window)):
        return np.nan, 0
    peak_idx = int(np.nanargmax(corr_window))
    return float(corr_window[peak_idx]), int(lag_window[peak_idx])


def rank_candidate_theta_trials(trace_pack, power_percentile=75, max_lag_ms=150, require_hpc_leads=True):
    source_traces = np.asarray(trace_pack['source_traces'], dtype=float)
    target_traces = np.asarray(trace_pack['target_traces'], dtype=float)
    sfreq = float(trace_pack['sfreq'])
    max_lag_samples = max(1, int(round((max_lag_ms / 1000.0) * sfreq)))

    source_power = np.mean(source_traces ** 2, axis=1)
    target_power = np.mean(target_traces ** 2, axis=1)
    source_threshold = float(np.nanpercentile(source_power, power_percentile))
    target_threshold = float(np.nanpercentile(target_power, power_percentile))

    rows = []
    for idx in range(source_traces.shape[0]):
        peak_corr, peak_lag = _peak_cross_correlation(source_traces[idx], target_traces[idx], max_lag_samples)
        rows.append({
            'epoch_trial': int(trace_pack['trial_indices'][idx] + 1),
            'behavior_trial': int(trace_pack['trial_values'][idx]),
            'hpc_theta_power': float(source_power[idx]),
            'acc_theta_power': float(target_power[idx]),
            'peak_corr': float(peak_corr),
            'peak_lag_samples': int(peak_lag),
            'peak_lag_ms': float((peak_lag / sfreq) * 1000.0),
            'hpc_leads_acc': bool(peak_lag > 0),
            'passes_power': bool((source_power[idx] >= source_threshold) and (target_power[idx] >= target_threshold)),
        })

    ranked = pd.DataFrame(rows)
    if len(ranked) == 0:
        return ranked, {'fallback_used': False, 'reason': 'no_trials'}

    ranked = ranked[np.isfinite(ranked['peak_corr'])].copy()
    if len(ranked) == 0:
        return ranked, {'fallback_used': False, 'reason': 'no_finite_cross_correlation'}

    ranked['power_score'] = ranked[['hpc_theta_power', 'acc_theta_power']].apply(lambda row: np.mean(_safe_zscore(row.values)), axis=1)
    ranked['corr_score'] = _safe_zscore(ranked['peak_corr'].values)
    ranked['lead_bonus'] = ranked['hpc_leads_acc'].astype(float)
    ranked['selection_score'] = ranked['power_score'] + ranked['corr_score'] + (0.35 * ranked['lead_bonus'])

    power_filtered = ranked[ranked['passes_power']].copy()
    if len(power_filtered) == 0:
        power_filtered = ranked.copy()
        fallback_used = True
        reason = 'no_high_power_trials'
    else:
        fallback_used = False
        reason = 'ok'

    if require_hpc_leads:
        lead_filtered = power_filtered[power_filtered['hpc_leads_acc']].copy()
        if len(lead_filtered) > 0:
            power_filtered = lead_filtered
        else:
            fallback_used = True
            reason = 'no_hpc_leads_trials'

    ranked = power_filtered.sort_values(['selection_score', 'peak_corr', 'hpc_theta_power', 'acc_theta_power'], ascending=[False, False, False, False]).reset_index(drop=True)
    meta = {
        'fallback_used': fallback_used,
        'reason': reason,
        'source_power_threshold': source_threshold,
        'target_power_threshold': target_threshold,
        'max_lag_samples': max_lag_samples,
    }
    return ranked, meta

ROI_PLOT_COLORS = {
    'HPC': np.array([0.44314, 0.2902, 1.0]),
    'AMY': np.array([0.49412, 0.0, 0.90196]),
    'OFC': np.array([1.0, 0.75686, 0.41569]),
    'ACC': np.array([0.7098, 0.27059, 0.68235]),
    'dmPFC': np.array([0.40784, 0.01961, 0.0]),
    'dlPFC': np.array([1.0, 0.06275, 0.06275]),
}


def _style_theta_axis(ax, roi_label, times):
    axis_color = '#2f2f2f'
    ax.set_facecolor('white')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(axis_color)
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(axis='both', colors=axis_color, labelsize=11, width=1.0, length=3)
    ax.axhline(0, color=axis_color, linestyle='--', linewidth=1.0, alpha=0.8)
    ax.axvline(0, color=axis_color, linestyle=':', linewidth=0.9, alpha=0.6)
    ax.set_xlim(times[0], times[-1])
    xticks = [tick for tick in (0.0, 0.5, 1.0, 1.5) if times[0] <= tick <= times[-1]]
    if len(xticks) > 0:
        ax.set_xticks(xticks)
    ax.text(
        0.01,
        0.07,
        roi_label,
        transform=ax.transAxes,
        color=axis_color,
        fontsize=15,
        ha='left',
        va='center',
    )


def plot_trialwise_theta_example(subj_id, roi_source, roi_target, trace_pack, candidate_row, condition, event, savepath=None):
    epoch_trial = int(candidate_row['epoch_trial'])
    matches = np.where(trace_pack['trial_indices'] == (epoch_trial - 1))[0]
    if matches.size == 0:
        raise ValueError(f'Trial {epoch_trial} not found in trace pack')
    arr_idx = int(matches[0])

    times = np.asarray(trace_pack['times'], dtype=float)
    source_trace = _safe_zscore(trace_pack['source_traces'][arr_idx])
    target_trace = _safe_zscore(trace_pack['target_traces'][arr_idx])
    peak_lag_samples = int(candidate_row['peak_lag_samples'])
    peak_lag_ms = float(candidate_row['peak_lag_ms'])
    source_color = ROI_PLOT_COLORS.get(roi_source, np.array([0.2, 0.6, 1.0]))
    target_color = ROI_PLOT_COLORS.get(roi_target, np.array([0.9, 0.3, 0.5]))
    axis_color = '#2f2f2f'

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(12, 5.4),
        dpi=300,
        sharex=True,
        gridspec_kw={'hspace': 0.18},
        facecolor='white',
    )

    traces = [
        (axes[0], roi_source, source_trace, source_color),
        (axes[1], roi_target, target_trace, target_color),
    ]
    y_extent = np.nanmax(np.abs(np.concatenate([source_trace, target_trace])))
    if (not np.isfinite(y_extent)) or y_extent == 0:
        y_extent = 1.0
    y_margin = 0.18 * y_extent

    for ax, roi_label, trace, color in traces:
        _style_theta_axis(ax, roi_label=roi_label, times=times)
        ax.plot(times, trace, color=color, linewidth=1.8, alpha=0.95)
        ax.set_ylim(-(y_extent + y_margin), y_extent + y_margin)

    shifted_trace = None
    shifted_ax = None
    shifted_color = '#7a7a7a'
    shifted_label = None
    if peak_lag_samples > 0:
        shifted_trace = np.full_like(source_trace, np.nan)
        shifted_trace[peak_lag_samples:] = source_trace[:-peak_lag_samples]
        shifted_label = f'{roi_source} shifted +{peak_lag_ms:.0f} ms'
        shifted_ax = axes[1]
    elif peak_lag_samples < 0:
        lag_abs = abs(peak_lag_samples)
        shifted_trace = np.full_like(target_trace, np.nan)
        shifted_trace[lag_abs:] = target_trace[:-lag_abs]
        shifted_label = f'{roi_target} shifted +{abs(peak_lag_ms):.0f} ms'
        shifted_ax = axes[0]
    if shifted_trace is not None:
        shifted_ax.plot(times, shifted_trace, linestyle='--', linewidth=1.2, color=shifted_color, alpha=0.85)
        shifted_ax.text(
            0.985,
            0.9,
            shifted_label,
            transform=shifted_ax.transAxes,
            color=shifted_color,
            fontsize=8.5,
            ha='right',
            va='center',
        )

    axes[0].set_xticklabels([])
    axes[0].set_ylabel('filtered\nLFP (z)', color=axis_color, fontsize=12)
    axes[1].set_ylabel('filtered\nLFP (z)', color=axis_color, fontsize=12)
    axes[1].set_xlabel(f'time since {event.replace("_", " ")} (s)', color=axis_color, fontsize=15, labelpad=12)

    fig.text(
        0.125,
        0.97,
        f'{subj_id}  {roi_source}->{roi_target}  |  {condition}  |  hemi={trace_pack["hemisphere"]}',
        color=axis_color,
        fontsize=12,
        fontweight='bold',
        ha='left',
        va='top',
    )
    fig.text(
        0.125,
        0.935,
        f'behavior trial {int(candidate_row["behavior_trial"])} | epoch {epoch_trial} | '
        f'peak corr={candidate_row["peak_corr"]:.3f} | lag={peak_lag_ms:.1f} ms | '
        f'HPC leads ACC={bool(candidate_row["hpc_leads_acc"])} | plot sfreq={trace_pack["sfreq"]:.1f} Hz',
        color=axis_color,
        fontsize=10,
        ha='left',
        va='top',
    )

    saved_file = None
    if savepath is not None:
        fname = (
            f'{subj_id}_{event}_{fmin_tag}_{roi_source}_{roi_target}_{condition}_'
            f'theta_example_trial{int(candidate_row["behavior_trial"])}_'
            f'epoch{epoch_trial}_{trace_pack["hemisphere"]}.pdf'
        )
        saved_file = os.path.join(plot_savepath, fname)
        fig.savefig(saved_file, dpi=300, bbox_inches='tight', facecolor='white')
    return fig, saved_file




In [ ]:
# Trialwise theta-LFP example plots for HPC -> ACC
theta_example_candidates = pd.DataFrame()
theta_example_context = pd.DataFrame()
theta_example_meta = None
theta_example_trace_pack = None
theta_example_native_epoch_data = None
theta_example_figures = []

if not RUN_SINGLE_SUBJECT_SMOKE:
    print('Trialwise theta example requires RUN_SINGLE_SUBJECT_SMOKE = True.')
elif smoke_pack is None or smoke_pack.get('result') is None:
    print('Smoke result is unavailable; rerun the smoke cell before generating examples.')
elif (SMOKE_ROI_SOURCE, SMOKE_ROI_TARGET) != EXAMPLE_PAIR:
    print('Smoke ROI pair does not match EXAMPLE_PAIR. Update SMOKE_ROI_SOURCE/SMOKE_ROI_TARGET or EXAMPLE_PAIR.')
else:
    theta_example_context = _summarize_gc_example_context(
        smoke_pack['result'],
        preferred_condition=EXAMPLE_CONDITION,
    )

    if len(theta_example_context) == 0 and EXAMPLE_CONDITION is not None:
        print(f'No successful GC context for condition={EXAMPLE_CONDITION}; falling back to any available RPE category.')
        theta_example_context = _summarize_gc_example_context(smoke_pack['result'], preferred_condition=None)

    if len(theta_example_context) == 0:
        print('No valid hemisphere/condition contexts available for theta example plotting.')
    else:
        display(theta_example_context[['hemisphere', 'condition', 'metric_key', 'peak_gc', 'mean_gc', 'n_trials']])
        selected_context = theta_example_context.iloc[0]
        theta_example_native_epoch_data = load_native_example_epoch_data(
            subj_id=SMOKE_SUBJECT,
            roi_source=EXAMPLE_PAIR[0],
            roi_target=EXAMPLE_PAIR[1],
            event=ANALYSIS['event'],
            day=ANALYSIS['day'],
            tmin=ANALYSIS['tmin'],
            tmax=ANALYSIS['tmax'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
        )
        print(
            'GC sfreq=', float(smoke_pack['result']['epoch_data']['sfreq']),
            '| plotting sfreq=', float(theta_example_native_epoch_data['sfreq']),
            '| original sfreq=', float(theta_example_native_epoch_data.get('original_sfreq', theta_example_native_epoch_data['sfreq']))
        )
        theta_example_trace_pack = extract_theta_pair_traces(
            theta_example_native_epoch_data,
            hemisphere=selected_context['hemisphere'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
            trial_indices=selected_context['trial_indices'],
            trial_values=selected_context['trial_values'],
            time_window=EXAMPLE_TIME_WINDOW,
        )
        theta_example_candidates, theta_example_meta = rank_candidate_theta_trials(
            theta_example_trace_pack,
            power_percentile=EXAMPLE_POWER_PERCENTILE,
            max_lag_ms=EXAMPLE_MAX_LAG_MS,
            require_hpc_leads=EXAMPLE_REQUIRE_HPC_LEADS,
        )

        if len(theta_example_candidates) == 0:
            print('No candidate trials passed the theta example ranking step.')
        else:
            if theta_example_meta.get('fallback_used', False):
                print('Lead/power criterion relaxed:', theta_example_meta.get('reason'))
            print('Theta candidate thresholds:',
                  'HPC >=', round(theta_example_meta['source_power_threshold'], 4),
                  '| ACC >=', round(theta_example_meta['target_power_threshold'], 4),
                  '| max lag samples =', theta_example_meta['max_lag_samples'])
            display(theta_example_candidates.head(EXAMPLE_TOP_K_CANDIDATES))

            for _, candidate_row in theta_example_candidates.head(EXAMPLE_TOP_K_CANDIDATES).iterrows():
                fig, saved_file = plot_trialwise_theta_example(
                    subj_id=SMOKE_SUBJECT,
                    roi_source=EXAMPLE_PAIR[0],
                    roi_target=EXAMPLE_PAIR[1],
                    trace_pack=theta_example_trace_pack,
                    candidate_row=candidate_row,
                    condition=selected_context['condition'],
                    event=ANALYSIS['event'],
                    savepath=analysis_paths['base_dir'],
                )
                theta_example_figures.append(fig)
                if saved_file is not None:
                    print('Saved theta example:', saved_file)
                plt.show()



In [ ]:
behavior_merge_df = combined_df if {'hits', 'memory_outcome'}.issubset(combined_df.columns) else rpe_behavior_df
# Full batch run across ROI pairs / subjects
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'all_gc_df' in analysis_state:
    all_gc_df = analysis_state.get('all_gc_df', pd.DataFrame()).copy()
    cache_log_df = analysis_state.get('cache_log_df', pd.DataFrame()).copy()
    print('Loaded batch results from in-memory registry for', analysis_tag)
elif (not OVERWRITE_GC_RPE_DF_CACHE) and os.path.exists(analysis_paths['all_gc_csv']):
    all_gc_df = pd.read_csv(analysis_paths['all_gc_csv'])
    cache_log_df = pd.read_csv(analysis_paths['cache_log_csv']) if os.path.exists(analysis_paths['cache_log_csv']) else pd.DataFrame()
    print('Loaded batch results from analysis bundle:', analysis_paths['all_gc_csv'])
else:
    all_rows = []
    cache_log_rows = []

    for roi_source, roi_target in ROI_PAIRS:
        result_key = f'{roi_source}_{roi_target}_{analysis_tag}'
        n_with_results = 0

        for subj in subjects:
            _, _, meta = run_or_load_subject_gc_rpe_df(
                subj_id=subj,
                roi_source=roi_source,
                roi_target=roi_target,
                base_dir=base_dir,
                learn_df=rpe_behavior_df,
                cache_dir=GC_RPE_DF_CACHE_DIR,
                overwrite_cache=OVERWRITE_GC_RPE_DF_CACHE,
                return_result_dict=False,
                event=ANALYSIS['event'],
                day=ANALYSIS['day'],
                tmin=ANALYSIS['tmin'],
                tmax=ANALYSIS['tmax'],
                target_sfreq=ANALYSIS['target_sfreq'],
                fmin=ANALYSIS['fmin'],
                fmax=ANALYSIS['fmax'],
                gc_n_lags=ANALYSIS['gc_n_lags'],
                n_freqs=ANALYSIS['n_freqs'],
                min_trials=ANALYSIS['min_trials'],
                use_surrogates=USE_SURROGATES,
                n_surrogates=N_SURROGATES_BATCH,
                surrogate_tail=SURROGATE_TAIL,
                random_state=SURROGATE_RANDOM_STATE,
                n_jobs_surrogates=N_JOBS_SURROGATES_BATCH,
                surrogate_parallel_backend=SURROGATE_PARALLEL_BACKEND,
                store_surrogates=False,
                verbose=False,
            )

            cache_file = meta.get('cache_file', '')
            subj_df = load_gc_rpe_df_cache(cache_file) if cache_file else None

            cache_log_rows.append({
                'participant': subj,
                'roi_source': roi_source,
                'roi_target': roi_target,
                'cache_hit': bool(meta.get('cache_hit', False)),
                'cache_file': cache_file,
                'loaded_from_cache': bool(subj_df is not None and len(subj_df) > 0),
            })

            if subj_df is None or len(subj_df) == 0:
                continue

            n_with_results += 1
            all_rows.append(subj_df)

        print(result_key, 'subjects with results:', n_with_results)

    all_gc_df = pd.concat(all_rows, ignore_index=True) if len(all_rows) > 0 else pd.DataFrame()
    cache_log_df = pd.DataFrame(cache_log_rows)
    all_gc_df.to_csv(analysis_paths['all_gc_csv'], index=False)
    cache_log_df.to_csv(analysis_paths['cache_log_csv'], index=False)
    print('Saved analysis bundle:', analysis_paths['all_gc_csv'])

# Enrich all_gc_df with trial-level behavior columns (including hit/miss) when available.
if len(all_gc_df) > 0:
    enrich_cols = {'hits', 'memory_outcome'}
    if not enrich_cols.issubset(all_gc_df.columns):
        merged_rows = []
        for (roi_source, roi_target), pair_df in all_gc_df.groupby(['roi_source', 'roi_target']):
            pair_merged = merge_with_behavior_rpe(pair_df.copy(), behavior_merge_df, event=ANALYSIS['event'])
            merged_rows.append(pair_merged)
        all_gc_df = pd.concat(merged_rows, ignore_index=True) if len(merged_rows) > 0 else all_gc_df

analysis_state['all_gc_df'] = all_gc_df.copy()
analysis_state['cache_log_df'] = cache_log_df.copy()
print('Total exported rows:', len(all_gc_df))
if len(all_gc_df) > 0 and 'gc_net_z' in all_gc_df.columns:
    print('Batch gc_net_z finite rows:', int(all_gc_df['gc_net_z'].notna().sum()))
if len(cache_log_df) > 0:
    print('Cache hits:', int(cache_log_df['cache_hit'].sum()), '/', len(cache_log_df))
    print('Loaded rows from cache files:', int(cache_log_df['loaded_from_cache'].sum()), '/', len(cache_log_df))
    display(cache_log_df.head())


In [ ]:
all_gc_df


In [ ]:
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'all_gc_df' in analysis_state:
    all_gc_df = analysis_state['all_gc_df'].copy()
else:
    all_gc_df = pd.read_csv(analysis_paths['all_gc_csv'])
    analysis_state['all_gc_df'] = all_gc_df.copy()
print('Loaded all_gc_df rows:', len(all_gc_df), '| from', analysis_paths['all_gc_csv'])


In [ ]:
all_gc_df

In [ ]:
import seaborn as sns
sns.lineplot(data=all_gc_df[(all_gc_df.roi_target == 'ACC')], 
x='time', y='gc_net_z', hue='rpe_cat', errorbar='se')


In [ ]:
# # analysis_tag = _build_analysis_tag(ANALYSIS, ROI_PAIRS)
# analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
# if 'all_gc_df' in analysis_state:
#     all_gc_df = analysis_state['all_gc_df'].copy()
# else:
#     all_gc_df = pd.read_csv(analysis_paths['all_gc_csv'])
#     analysis_state['all_gc_df'] = all_gc_df.copy()
# print('Loaded all_gc_df rows:', len(all_gc_df), '| from', analysis_paths['all_gc_csv'])


In [ ]:
behavior_merge_df = combined_df if {'hits', 'memory_outcome'}.issubset(combined_df.columns) else rpe_behavior_df
# Export RPE outputs and behavior-merged files
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if len(all_gc_df) == 0:
    print('No GC rows to export.')
    merged_gc_df = pd.DataFrame()
elif 'merged_gc_df' in analysis_state:
    merged_gc_df = analysis_state['merged_gc_df'].copy()
    print('Loaded merged_gc_df from in-memory registry for', analysis_tag)
elif os.path.exists(analysis_paths['merged_gc_csv']):
    merged_gc_df = pd.read_csv(analysis_paths['merged_gc_csv'])
    print('Loaded merged_gc_df from analysis bundle:', analysis_paths['merged_gc_csv'])
else:
    all_gc_df.to_csv(analysis_paths['all_gc_csv'], index=False)
    print('Saved:', analysis_paths['all_gc_csv'])

    merged_rows = []
    has_hitmiss_cols = {'hits', 'memory_outcome'}.issubset(all_gc_df.columns)
    for (roi_source, roi_target), pair_df in all_gc_df.groupby(['roi_source', 'roi_target']):
        merged = pair_df.copy() if has_hitmiss_cols else merge_with_behavior_rpe(pair_df.copy(), behavior_merge_df, event=ANALYSIS['event'])
        merged_rows.append(merged)
        pair_file = os.path.join(analysis_paths['base_dir'], f'{roi_source}_{roi_target}_mvgc_rpe_merged.csv')
        merged.to_csv(pair_file, index=False)
    merged_gc_df = pd.concat(merged_rows, ignore_index=True) if len(merged_rows) > 0 else pd.DataFrame()
    merged_gc_df.to_csv(analysis_paths['merged_gc_csv'], index=False)
    print('Saved merged bundle:', analysis_paths['merged_gc_csv'])

analysis_state['all_gc_df'] = all_gc_df.copy()
analysis_state['merged_gc_df'] = merged_gc_df.copy()
if len(merged_gc_df) > 0:
    print('Merged rows:', len(merged_gc_df))
    display(merged_gc_df.head())


In [ ]:
# Mixed effects: gc_net_z ~ rpe_cat (random intercept: participant), ACC targets, cue_on only.
acc_cue_df = all_gc_df.copy()
if 'event' in acc_cue_df.columns:
    acc_cue_df = acc_cue_df[acc_cue_df['event'].eq('cue_on')]
acc_cue_df = acc_cue_df[acc_cue_df['roi_target'].eq('ACC')].copy()

model_df = acc_cue_df.copy()
model_df['target_roi'] = model_df['roi_target']

if ('gc_net_z' not in model_df.columns) or (not model_df['gc_net_z'].notna().any()):
    _gb = ['participant', 'roi_target']
    if 'event' in model_df.columns and model_df['event'].nunique() > 1:
        _gb = ['participant', 'roi_target', 'event']
    model_df['gc_net_z'] = model_df.groupby(_gb)['gc_net'].transform(
        lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
    )

print(
    'ACC / cue_on rows:', len(model_df),
    '| participants:', model_df['participant'].nunique(),
    '| finite gc_net_z:', int(model_df['gc_net_z'].notna().sum()),
)

model_out_acc_cue = fit_mixed_effects_rpe(model_df, reference='neutral')
display(model_out_acc_cue['stats_df'])
for target, res in model_out_acc_cue['model_results'].items():
    print(f'--- {target} ---')
    if hasattr(res, 'summary'):
        print(res.summary())
    else:
        print(res)


In [ ]:
import statsmodels.formula.api as smf
formula = f"gc_net_z ~ C(rpe_cat, Treatment('neg'))"
model = smf.mixedlm(formula, data=model_df, groups=model_df["participant"])
fit = model.fit(method="lbfgs", maxiter=500)
print(fit.summary())

## Figure 5D:

In [ ]:
# Aggregate plots + mixed-effects models (RPE)
# Load both events for consolidated left=feedback_start, right=cue_on plot (same axes).
analysis_state = ANALYSIS_RESULTS.setdefault(analysis_tag, {})
if 'merged_gc_df' in analysis_state:
    merged_gc_df = analysis_state['merged_gc_df'].copy()
else:
    merged_gc_df = pd.DataFrame()

event_specs = [
    ('feedback_start', 1.5),
    ('cue_on', 1.0),
]
event_frames = []
for ev_name, tmax in event_specs:
    analysis_ev = dict(ANALYSIS)
    analysis_ev['event'] = ev_name
    analysis_ev['tmax'] = tmax
    tag_ev = _build_analysis_tag(analysis_ev, ROI_PAIRS)
    path_ev = get_analysis_paths(tag=tag_ev)['merged_gc_csv']
    if os.path.exists(path_ev):
        df_ev = pd.read_csv(path_ev)
        df_ev['event'] = ev_name
        event_frames.append(df_ev)

combined_for_plot = pd.concat(event_frames, ignore_index=True) if len(event_frames) > 0 else pd.DataFrame()

if len(merged_gc_df) == 0 and len(combined_for_plot) == 0:
    print('No merged data for aggregate/model plotting.')
else:
    source_roi = 'HPC'
    source_paths = get_analysis_paths(source_roi=source_roi)
    agg_df = merged_gc_df[merged_gc_df['roi_source'] == source_roi].copy() if len(merged_gc_df) > 0 else pd.DataFrame()

    if len(agg_df) > 0:
        fig_agg = plot_aggregate_rpe(
            agg_df,
            source_roi=source_roi,
            savepath=source_paths['base_dir'],
            event=ANALYSIS['event'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
        )

        if fig_agg is not None:
            plt.show()

    # Event-split mixed-effects plot: left=feedback_start, right=cue_on, shared axes.
    if len(combined_for_plot) > 0:
        plot_df = combined_for_plot[combined_for_plot['roi_source'] == source_roi].copy()
        plot_df['target_roi'] = plot_df['roi_target']
        if ('gc_net_z' not in plot_df.columns) or (not plot_df['gc_net_z'].notna().any()):
            plot_df['gc_net_z'] = plot_df.groupby(['participant', 'roi_target', 'event'])['gc_net'].transform(
                lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
            )
        model_out_plot = {'model_df': plot_df, 'stats_df': pd.DataFrame()}
        fig_mixed = plot_mixed_effects_rpe(
            model_out_plot,
            source_roi=source_roi,
            savepath=source_paths['base_dir'],
            event=ANALYSIS['event'],
            fmin=ANALYSIS['fmin'],
            fmax=ANALYSIS['fmax'],
            ylim=(-2, 2),
        )

        if fig_mixed is not None:
            plt.show()
    else:
        # Fallback: single-event from merged_gc_df (right panel will show "no data" if plot expects event column).
        model_df = agg_df.copy()
        model_df['target_roi'] = model_df['roi_target']
        if len(model_df) > 0:
            if ('gc_net_z' not in model_df.columns) or (not model_df['gc_net_z'].notna().any()):
                model_df['gc_net_z'] = model_df.groupby(['participant', 'roi_target'])['gc_net'].transform(
                    lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
                )
            model_out = fit_mixed_effects_rpe(model_df, reference='neutral')
            fig_mixed = plot_mixed_effects_rpe(
                model_out,
                source_roi=source_roi,
                savepath=source_paths['base_dir'],
                event=ANALYSIS['event'],
                fmin=ANALYSIS['fmin'],
                fmax=ANALYSIS['fmax'],
            )
            fig_mixed.get_axes()[0].set_ylim(-2, 2)
            fig_mixed.get_axes()[1].set_ylim(-2, 2)
            sns.despine()
            if fig_mixed is not None:
                plt.show()
            analysis_state.setdefault('model_out', {})[source_roi] = model_out
            if len(model_out.get('stats_df', pd.DataFrame())) > 0:
                stats_file = source_paths['stats_csv']
                model_out['stats_df'].to_csv(stats_file, index=False)
                print('Saved mixed-model stats:', stats_file)
                display(model_out['stats_df'].head())


In [ ]:
import statsmodels.formula.api as smf
formula = f"gc_net_z ~ C(rpe_cat, Treatment('pos'))"
model = smf.mixedlm(formula, data=model_df, groups=model_df["participant"])
fit = model.fit(method="lbfgs", maxiter=500)
print(fit.summary())

In [ ]:
# Bar + strip: cue_on only, neg / neutral / pos RPE (styling aligned with plot_mixed_effects_rpe).
_cfp = globals().get('combined_for_plot', pd.DataFrame())
_src = globals().get('source_roi', 'HPC')

if len(_cfp) == 0:
    print(
        'Cue neutral/pos bar+strip: combined_for_plot is empty. '
        'Load both merged_gc_csv files (feedback_start and cue_on) so the aggregate cell can build it.'
    )
elif 'event' not in _cfp.columns:
    print('Cue neutral/pos bar+strip: combined_for_plot has no event column.')
else:
    plot_df = _cfp[(_cfp['event'].eq('cue_on')) & (_cfp['roi_source'].eq(_src))].copy()
    if len(plot_df) == 0:
        print('Cue neutral/pos bar+strip: no rows for cue_on and source_roi=', _src)
    else:
        plot_df['target_roi'] = plot_df['roi_target']
        if ('gc_net_z' not in plot_df.columns) or (not plot_df['gc_net_z'].notna().any()):
            plot_df['gc_net_z'] = plot_df.groupby(['participant', 'roi_target', 'event'])['gc_net'].transform(
                lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
            )
        _rpe_map = {'negative': 'neg', 'neg': 'neg', 'neutral': 'neutral', 'positive': 'pos', 'pos': 'pos'}
        plot_df['rpe_cat'] = plot_df['rpe_cat'].astype(str).str.strip().str.lower().map(_rpe_map)
        plot_df = plot_df[plot_df['rpe_cat'].isin(['neg', 'neutral', 'pos'])].copy()
        if len(plot_df) == 0:
            print('Cue neutral/pos bar+strip: no rows after filtering to neg/neutral/pos.')
        else:
            cats = ['neg', 'neutral', 'pos']
            colors = {'neg': '#d62728', 'neutral': '#7f7f7f', 'pos': '#2ca02c'}
            target_order = sorted(plot_df['target_roi'].dropna().unique())
            fig, ax = plt.subplots(1, 1, figsize=(7, 5))
            x = np.arange(len(target_order))
            n_cats = len(cats)
            # Center n_cats bars on each tick: offsets from -(n-1)/2 to +(n-1)/2 (i-0.5 spacing is only for 2 bars).
            bar_width = 0.22
            means = plot_df.groupby(['target_roi', 'rpe_cat'])['gc_net_z'].agg(['mean', 'sem']).reset_index()
            for i, cat in enumerate(cats):
                vals = means[means['rpe_cat'] == cat].set_index('target_roi').reindex(target_order)
                ax.bar(
                    x + (i - (n_cats - 1) / 2) * bar_width,
                    vals['mean'],
                    bar_width,
                    yerr=vals['sem'],
                    color=colors[cat],
                    alpha=0.85,
                    capsize=2,
                    label=cat,
                )
            subj_means = (
                plot_df.groupby(['participant', 'target_roi', 'rpe_cat'])['gc_net_z']
                .mean()
                .reset_index()
            )
            sns.stripplot(
                data=subj_means,
                x='target_roi',
                y='gc_net_z',
                hue='rpe_cat',
                order=target_order,
                hue_order=cats,
                dodge=True,
                palette=colors,
                alpha=0.6,
                edgecolor='black',
                linewidth=0.3,
                size=3,
                ax=ax,
                zorder=10,
            )
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()
            ax.axhline(0, color='gray', linewidth=1)
            ax.set_xticks(x)
            ax.set_xticklabels(target_order)
            ax.set_ylabel('Mean gc_net_z')
            ax.set_title(f'cue_on | neg + neutral + pos | {_src}→targets')
            ax.legend(loc='upper right')
            plt.suptitle(f'Bar + strip: {_src}→targets (RPE)', y=1.02)
            plt.tight_layout()
            _sp = globals().get('source_paths')
            if isinstance(_sp, dict) and _sp.get('base_dir'):
                os.makedirs(_sp['base_dir'], exist_ok=True)
                fmin, fmax = ANALYSIS.get('fmin'), ANALYSIS.get('fmax')
                band_tag = f"_{fmin}p{fmax}Hz" if fmin is not None and fmax is not None else ''
                _pdf = os.path.join(_sp['base_dir'], f'mixed_strip_cue_on_neutral_pos_{_src}{band_tag}.pdf')
                fig.savefig(_pdf, dpi=300, bbox_inches='tight', facecolor='white')
                print('Saved:', _pdf)
            plt.show()


In [ ]:
fig_mixed.get_axes()[0].set_ylim(-2, 2)

In [ ]:
get_analysis_paths(source_roi='ACC')

In [ ]:
source_paths

In [ ]:
# Cross-event task-period mixed-effects helpers
import statsmodels.formula.api as smf


def build_cross_event_analysis_tag(event, fmin, fmax, roi_pairs=None, analysis=None):
    analysis_local = dict(ANALYSIS if analysis is None else analysis)
    analysis_local['event'] = event
    analysis_local['fmin'] = fmin
    analysis_local['fmax'] = fmax
    return _build_analysis_tag(analysis_local, ROI_PAIRS if roi_pairs is None else roi_pairs)


def load_cross_event_merged_gc(event, fmin, fmax, roi_pairs=None, analysis=None):
    analysis_local = dict(ANALYSIS if analysis is None else analysis)
    analysis_local['event'] = event
    analysis_local['fmin'] = fmin
    analysis_local['fmax'] = fmax
    tag = build_cross_event_analysis_tag(event, fmin, fmax, roi_pairs=roi_pairs, analysis=analysis_local)
    analysis_state = ANALYSIS_RESULTS.get(tag, {})
    if 'merged_gc_df' in analysis_state:
        merged_gc_df = analysis_state['merged_gc_df'].copy()
        bundle_path = get_analysis_paths(tag=tag)['merged_gc_csv']
        return merged_gc_df, bundle_path, tag

    bundle_path = get_analysis_paths(tag=tag)['merged_gc_csv']
    if not os.path.exists(bundle_path):
        raise FileNotFoundError(
            'Missing merged RPE GC bundle for '
            f'event={event}, fmin={fmin}, fmax={fmax}. '            f'Expected file: {bundle_path}. '            'Run the single-event RPE notebook analysis for that configuration first.'
        )
    merged_gc_df = pd.read_csv(bundle_path)
    ANALYSIS_RESULTS.setdefault(tag, {})['merged_gc_df'] = merged_gc_df.copy()
    return merged_gc_df, bundle_path, tag


def prepare_task_period_model_df(band_label, fmin, fmax, roi_pairs=None, rpe_keep=('neutral', 'pos')):
    roi_pairs = ROI_PAIRS if roi_pairs is None else roi_pairs
    event_specs = [
        ('feedback', 'feedback_start'),
        ('cue', 'cue_on'),
    ]
    event_frames = []
    bundle_info = []

    for task_condition, event_name in event_specs:

        analysis_dict = {
            'event': event_name,
            'day': 'Day1',
            'tmin': 0.0,
            'tmax': 1.0 if event_name == 'cue_on' else 1.5,
            'target_sfreq': 50,
            'fmin': fmin,
            'fmax': fmax,
            'gc_n_lags': 10,
            'n_freqs': None,
            'min_trials': 5,
        }

        merged_gc_df, bundle_path, tag = load_cross_event_merged_gc(
            event=event_name,
            fmin=fmin,
            fmax=fmax,
            roi_pairs=roi_pairs,
            analysis=analysis_dict,
        )
        bundle_info.append({
            'task_condition': task_condition,
            'event': event_name,
            'bundle_path': bundle_path,
            'analysis_tag': tag,
        })
        if len(merged_gc_df) == 0:
            continue
        event_df = merged_gc_df.copy()
        event_df['task_condition'] = task_condition
        event_df['event'] = event_name
        event_df['band_label'] = band_label
        event_frames.append(event_df)

    model_df = pd.concat(event_frames, ignore_index=True) if len(event_frames) > 0 else pd.DataFrame()
    if len(model_df) == 0:
        return model_df, pd.DataFrame(bundle_info)

    required_cols = ['participant', 'roi_source', 'roi_target', 'rpe_cat', 'task_condition', 'gc_net_z']
    missing_cols = [col for col in required_cols if col not in model_df.columns]
    if missing_cols:
        raise KeyError(f'Missing required columns for task-period model: {missing_cols}')

    model_df = model_df[model_df['rpe_cat'].isin(list(rpe_keep))].copy()
    model_df = model_df[model_df['roi_source'].notna() & model_df['roi_target'].notna()].copy()
    model_df = model_df[model_df['gc_net_z'].notna()].copy()

    grouping_cols = [
        'participant',
        'roi_source',
        'roi_target',
        'task_condition',
        'rpe_cat',
        'band_label',
    ]
    if 'hemisphere' in model_df.columns:
        grouping_cols.append('hemisphere')

    averaged_df = (
        model_df
        .groupby(grouping_cols, dropna=False)
        .agg(
            gc_net_z=('gc_net_z', 'mean'),
            gc_net=('gc_net', 'mean') if 'gc_net' in model_df.columns else ('gc_net_z', 'mean'),
            n_timepoints=('gc_net_z', 'size'),
        )
        .reset_index()
    )

    averaged_df['rpe_cat'] = pd.Categorical(averaged_df['rpe_cat'], categories=list(rpe_keep), ordered=True)
    averaged_df['task_condition'] = pd.Categorical(averaged_df['task_condition'], categories=['feedback', 'cue'], ordered=True)
    return averaged_df, pd.DataFrame(bundle_info)


def _clean_task_period_term_name(term_name, reference_rpe='neutral', reference_task='feedback'):
    clean_name = term_name.replace(f", Treatment(reference='{reference_rpe}')", '')
    clean_name = clean_name.replace(f", Treatment(reference='{reference_task}')", '')
    return clean_name


def fit_task_period_mixed_effects(model_df, reference_rpe='neutral', reference_task='feedback'):
    stats_rows = []
    failures = []
    fitted_models = {}
    formula = (
        f"gc_net_z ~ C(rpe_cat, Treatment(reference='{reference_rpe}')) * "
        f"C(task_condition, Treatment(reference='{reference_task}'))"
    )

    if len(model_df) == 0:
        return {
            'stats_df': pd.DataFrame(),
            'failures_df': pd.DataFrame(),
            'models': {},
            'formula': formula,
        }

    for (band_label, roi_source, roi_target), pair_df in model_df.groupby(['band_label', 'roi_source', 'roi_target']):
        pair_df = pair_df.copy()
        n_rows = len(pair_df)
        n_participants = pair_df['participant'].nunique()
        level_counts = {
            'rpe_levels': pair_df['rpe_cat'].nunique(),
            'task_levels': pair_df['task_condition'].nunique(),
        }
        if n_rows < 8 or n_participants < 2 or level_counts['rpe_levels'] < 2 or level_counts['task_levels'] < 2:
            failures.append({
                'band_label': band_label,
                'roi_source': roi_source,
                'roi_target': roi_target,
                'n_rows': n_rows,
                'n_participants': n_participants,
                'reason': 'Insufficient rows or missing factor levels for mixed-effects fit.',
            })
            continue

        try:
            model = smf.mixedlm(formula, data=pair_df, groups=pair_df['participant'])
            result = model.fit(reml=False, method='lbfgs', maxiter=200, disp=False)
            fitted_models[(band_label, roi_source, roi_target)] = result
            conf_df = result.conf_int()
            fixed_terms = list(model.exog_names)
            interaction_raw = (
                f"C(rpe_cat, Treatment(reference='{reference_rpe}'))[T.pos]:"
                f"C(task_condition, Treatment(reference='{reference_task}'))[T.cue]"
            )

            for term_name in fixed_terms:
                stats_rows.append({
                    'band_label': band_label,
                    'roi_source': roi_source,
                    'roi_target': roi_target,
                    'n_rows': n_rows,
                    'n_participants': n_participants,
                    'formula': formula,
                    'term': term_name,
                    'term_clean': _clean_task_period_term_name(term_name, reference_rpe, reference_task),
                    'is_interaction_pos_cue': term_name == interaction_raw,
                    'coef': result.params.get(term_name, np.nan),
                    'std_err': result.bse.get(term_name, np.nan),
                    'stat': result.tvalues.get(term_name, np.nan),
                    'p_value': result.pvalues.get(term_name, np.nan),
                    'ci_low': conf_df.loc[term_name, 0] if term_name in conf_df.index else np.nan,
                    'ci_high': conf_df.loc[term_name, 1] if term_name in conf_df.index else np.nan,
                })
        except Exception as exc:
            failures.append({
                'band_label': band_label,
                'roi_source': roi_source,
                'roi_target': roi_target,
                'n_rows': n_rows,
                'n_participants': n_participants,
                'reason': f'{type(exc).__name__}: {exc}',
            })

    stats_df = pd.DataFrame(stats_rows)
    failures_df = pd.DataFrame(failures)
    if len(stats_df) > 0:
        stats_df = stats_df.sort_values(['band_label', 'roi_source', 'roi_target', 'term']).reset_index(drop=True)
    return {
        'stats_df': stats_df,
        'failures_df': failures_df,
        'models': fitted_models,
        'formula': formula,
    }


def build_task_period_signature(roi_pairs=None):
    roi_pairs = ROI_PAIRS if roi_pairs is None else roi_pairs
    roi_pairs_norm = _normalize_roi_pairs(roi_pairs)
    roi_hash = hashlib.sha1(json.dumps(roi_pairs_norm, sort_keys=True).encode('utf-8')).hexdigest()[:10]
    return (
        f"feedback-cue_{ANALYSIS['day']}"
        f"_t{_analysis_value_token(ANALYSIS['tmin'])}-{_analysis_value_token(ANALYSIS['tmax'])}"
        f"_sf{_analysis_value_token(ANALYSIS['target_sfreq'])}"
        f"_lag{_analysis_value_token(ANALYSIS['gc_n_lags'])}"
        f"_roi{roi_hash}"
    )



In [ ]:
# Cross-event task-period mixed-effects model: pos vs neutral by cue vs feedback
TASK_PERIOD_BANDS = [
    ('theta', 4, 8)
]
TASK_PERIOD_DIR = os.path.join(savepath, 'task_period_rpe_models', build_task_period_signature())
os.makedirs(TASK_PERIOD_DIR, exist_ok=True)

cross_event_inputs = []
cross_event_stats = []
cross_event_failures = []

for band_label, band_fmin, band_fmax in TASK_PERIOD_BANDS:
    print(f'\nBand: {band_label} ({band_fmin}-{band_fmax} Hz)')
    model_input_df, bundle_info_df = prepare_task_period_model_df(
        band_label=band_label,
        fmin=band_fmin,
        fmax=band_fmax,
        roi_pairs=ROI_PAIRS,
        rpe_keep=('neutral', 'pos'),
    )
    if len(bundle_info_df) > 0:
        display(bundle_info_df)
    if len(model_input_df) == 0:
        print('No task-period model rows available for this band.')
        continue

    input_csv = os.path.join(TASK_PERIOD_DIR, f'task_period_rpe_model_input_{band_label}_{band_fmin}-{band_fmax}Hz.csv')
    model_input_df.to_csv(input_csv, index=False)
    print('Saved model input:', input_csv)

    model_out = fit_task_period_mixed_effects(
        model_input_df,
        reference_rpe='neutral',
        reference_task='feedback',
    )
    stats_df = model_out['stats_df'].copy()
    failures_df = model_out['failures_df'].copy()

    if len(stats_df) > 0:
        stats_csv = os.path.join(TASK_PERIOD_DIR, f'task_period_rpe_mixed_effects_{band_label}_{band_fmin}-{band_fmax}Hz.csv')
        stats_df.to_csv(stats_csv, index=False)
        print('Saved model coefficients:', stats_csv)
        interaction_df = (
            stats_df[stats_df['is_interaction_pos_cue']]
            .sort_values(['p_value', 'roi_source', 'roi_target'])
            .reset_index(drop=True)
        )
        if len(interaction_df) > 0:
            print('Interaction rows: pos vs neutral x cue vs feedback')
            display(interaction_df)
        else:
            print('No explicit pos x cue interaction rows were returned for this band.')
        cross_event_stats.append(stats_df)

    if len(failures_df) > 0:
        failures_csv = os.path.join(TASK_PERIOD_DIR, f'task_period_rpe_mixed_effects_failures_{band_label}_{band_fmin}-{band_fmax}Hz.csv')
        failures_df.to_csv(failures_csv, index=False)
        print('Saved fit failures:', failures_csv)
        display(failures_df)
        cross_event_failures.append(failures_df)

    cross_event_inputs.append(model_input_df)

combined_stats_df = pd.concat(cross_event_stats, ignore_index=True) if len(cross_event_stats) > 0 else pd.DataFrame()
combined_failures_df = pd.concat(cross_event_failures, ignore_index=True) if len(cross_event_failures) > 0 else pd.DataFrame()

if len(combined_stats_df) > 0:
    combined_stats_csv = os.path.join(TASK_PERIOD_DIR, 'task_period_rpe_mixed_effects_all_bands.csv')
    combined_stats_df.to_csv(combined_stats_csv, index=False)
    print('Saved combined coefficients:', combined_stats_csv)
    combined_interactions_df = (
        combined_stats_df[combined_stats_df['is_interaction_pos_cue']]
        .sort_values(['band_label', 'p_value', 'roi_source', 'roi_target'])
        .reset_index(drop=True)
    )
    print('Combined interaction summary:')
    display(combined_interactions_df)
else:
    print('No cross-event mixed-effects results were produced.')

if len(combined_failures_df) > 0:
    combined_failures_csv = os.path.join(TASK_PERIOD_DIR, 'task_period_rpe_mixed_effects_failures_all_bands.csv')
    combined_failures_df.to_csv(combined_failures_csv, index=False)
    print('Saved combined failures:', combined_failures_csv)

